# 🏡 Metrocuadrado Property Scraper (Colab-Ready)
This notebook uses Selenium to extract detailed property listings from Metrocuadrado.com for rental listings.

In [ ]:
!pip install selenium
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
import sys
sys.path.insert(0, '/usr/lib/chromium-browser/chromedriver')

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time, re, json
import pandas as pd
from bs4 import BeautifulSoup

In [ ]:
def init_driver():
    options = Options()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    return webdriver.Chrome(options=options)

In [ ]:
def extract_json_from_hydrated_html(html):
    try:
        pattern = re.compile(r'self\.__next_f\.push\(\[1,"6:(.*?)"\]\)', re.DOTALL)
        match = pattern.search(html)
        if not match:
            return None
        raw_data = match.group(1)
        decoded = bytes(raw_data, "utf-8").decode("unicode_escape")
        inner_json_match = re.search(r'"data":\s*({.*?})\s*(?=,\s*\"?moduleIds\"?|})', decoded, re.DOTALL)
        if not inner_json_match:
            return None
        json_obj = json.loads(inner_json_match.group(1))
        return json_obj
    except:
        return None

In [ ]:
def extract_clean_info(data):
    return {
        "id": data.get("propertyId"),
        "titulo": data.get("title"),
        "precio": data.get("rentPrice") or data.get("salePrice"),
        "habitaciones": data.get("rooms"),
        "baños": data.get("bathrooms"),
        "garajes": data.get("garages"),
        "estrato": data.get("stratum"),
        "área": data.get("area"),
        "tipo": data.get("propertyType", {}).get("nombre"),
        "ciudad": data.get("city", {}).get("nombre"),
        "zona": data.get("zone", {}).get("nombre"),
        "barrio": data.get("neighborhood"),
        "estado": data.get("propertyState"),
        "comentario": data.get("comment"),
        "latitud": data.get("coordinates", {}).get("lat"),
        "longitud": data.get("coordinates", {}).get("lon"),
        "dirección": data.get("companyAddress"),
        "empresa": data.get("companyName"),
        "link": data.get("link"),
        "imágenes": [img["image"] for img in data.get("images", [])],
        "video": data.get("video")
    }

In [ ]:
def extract_property_info(driver, url):
    try:
        driver.get(url)
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        time.sleep(2)
        html = driver.page_source
        data = extract_json_from_hydrated_html(html)
        return extract_clean_info(data) if data else None
    except Exception as e:
        print(f"❌ Error in {url}: {e}")
        return None

In [ ]:
def collect_property_links(driver, base_url, pages=2):
    all_links = []
    for i in range(1, pages+1):
        url = f"{base_url}?pagina={i}"
        try:
            driver.get(url)
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "a")))
            links = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/inmueble/"]')
            page_links = [l.get_attribute("href") for l in links if l.get_attribute("href")]
            all_links.extend(page_links)
            print(f"✅ Page {i} scraped: {len(page_links)} links")
        except Exception as e:
            print(f"⚠️ Failed page {i}: {e}")
        time.sleep(1)
    return list(dict.fromkeys(all_links))

In [ ]:
# 🚀 Scraping metrocuadrado listings
driver = init_driver()
base_url = "https://www.metrocuadrado.com/arriendo/medellin"
listing_links = collect_property_links(driver, base_url, pages=2)

results = []
for link in listing_links:
    info = extract_property_info(driver, link)
    if info:
        results.append(info)
    time.sleep(0.5)

driver.quit()

df = pd.DataFrame(results)
df.to_csv("metrocuadrado_properties.csv", index=False)
df.head()